# Backtesting VaR

A VaR model is only useful if it is well-calibrated, meaning that the predicted loss quantiles are consistent with realised portfolio losses. Backtesting provides a formal statistical framework for evaluating whether observed VaR exceedances occur at the frequency implied by the model.

Under the Basel framework, banks are required to backtest internal VaR models using realised trading outcomes. Excessive or clustered exceedances may indicate model misspecification and can lead to higher regulatory capital requirements under the Basel traffic-light system.

We evaluate the three VaR models developed in Notebook 02:

- Historical Simulation (HS)
- Parametric Normal
- GARCH(1,1)-filtered Historical Simulation

using two complementary families of backtests.

**Exceedance-based tests** 

Exceedance-based tests count the number of days the realised loss exceeds the VaR forecast and test whether this frequency is consistent with the stated confidence level using:

- **z-test** compares the observed exceedance frequency with the theoretical rate
- **Kupiec LR$_{uc}$** likelihood ratio test tests unconditional coverage (whether the total number of exceedances is correct)
- **Christoffersen LR$_{cc}$** extends Kupiec by additionally testing independence of exceedances, since clustered exceptions indicate that risk forecasts fail to adapt to changing market conditions

**PIT-based tests** 

VaR uses only a single quantile of the forecast distribution. PIT-based methods evaluate the entire predictive distribution. For each forecast, the Probability Integral Transform (PIT) is defined as:
$$ u_t = F_t (r_t) $$
where $F(\cdot)$ denotes the forecast cumulative distribution function. If the model is correctly specified, the PIT sequence should be independent and uniformly distributed on $ [0,1] $.

We implement:

- **Kolmogorov-Smirnov (K-S) test** tests maximum deviation from Unif(0,1)
- **Anderson-Darling (A-D) test** is a weighted K-S with emphasis on tails
- **Cramér-von Mises (CvM) test** tests mean squared deviation from uniformity

All tests are applied to all three VaR models from Notebook 02, and repeated separately within each HMM regime (Expansion, Normal, and Stress).

In [21]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from arch import arch_model
from scipy import stats
from scipy.stats import chi2
from scipy.stats import kstest, anderson, cramervonmises

import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
np.random.seed(926)

ROOT = Path().resolve().parent
OUT_RETURNS = ROOT / 'data' / 'processed' / 'returns'
OUT_REGIMES = ROOT / 'data' / 'processed' / 'regimes'
OUT_RISK = ROOT / 'data' / 'processed' / 'risk_metrics'
OUT_BT = ROOT / 'data' / 'processed' / 'risk_metrics'

portfolio_return = pd.read_csv( OUT_RETURNS / 'portfolio_return.csv', index_col='date', parse_dates=True).squeeze()

risk_estimates = pd.read_csv(OUT_RISK / 'rolling_var_es.csv', index_col='date', parse_dates=True)

regime_labels = pd.read_csv(OUT_REGIMES / 'regime_labels.csv', index_col='date', parse_dates=True)['regime']


common_idx = risk_estimates.index
port_aligned = portfolio_return.reindex(common_idx)
regime_aligned = regime_labels.reindex(common_idx)

CONF_VAR = 0.99 # 99% VaR
CONF_ES  = 0.975 # 97.5% ES
p = 1 - CONF_VAR # expected exceedance probability = 1%

## Exceedance-Based Backtesting

### z-test

The z-test is the simplest exceedance-based backtesting procedure. It treats the number of VaR exceedances as a binomial random variable:

$$ N \sim \text{Bin}(T, p), \quad p = 1 - \alpha = 0.01 \quad \text{under } 99\% \text{ VaR} $$

where $T$ is the number of observations and $p$ is the theoretical exceedance probability implied by the VaR confidence level.

Under $H_0$ (model well-calibrated), the z-statistic is:

$$ z = \frac{N - pT}{\sqrt{p(1-p)T}} \sim \mathcal{N}(0,1) $$

We reject $H_0$ at 5% significance if $|z| > 1.96$. A positive z-statistic indicates more exceedances than expected (risk underestimation), while a negative z-statistic indicates fewer exceedances than expected (risk overestimation).

**Limitation**: The z-test only checks the *frequency* of exceedances, not their *timing*. Clustered exceptions, which signal volatility regime failure, are not detected.

In [3]:
def z_test(exceptions, n_obs, p=0.01, alpha=0.05):
    """
    Binomial proportion z-test for VaR exceedance rate.
    """
    actual_rate = exceptions / n_obs
    z_stat = (exceptions - p * n_obs) / np.sqrt(p * (1 - p) * n_obs)
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat))) # two-sided
    return {
        'N exceptions' : exceptions,
        'T obs' : n_obs,
        'Actual rate' : round(actual_rate * 100, 3),
        'Expected rate': round(p * 100, 3),
        'z-stat' : round(z_stat, 4),
        'p-value' : round(p_value, 4),
        'Reject H0' : 'Yes' if p_value < alpha else 'No',
    }

# ── Run z-test for all three methods ─────────────────────────────────────────
methods = {
    'Historical Simulation': 'hs_var99',
    'Parametric (Normal)' : 'param_var99',
    'GARCH(1,1)-filtered' : 'garch_var99',
}

print('z-test Results (H0: exceedance rate = 1.0%):')
print('=' * 65)

ztest_results = {}
for method, col in methods.items():
    var_series = risk_estimates[col].reindex(common_idx)
    losses = -port_aligned
    exceptions = int((losses > var_series).sum())
    result = z_test(exceptions, len(common_idx))
    ztest_results[method] = result
    print(f'\n{method}:')
    for k, v in result.items():
        print(f'  {k:<18}: {v}')

z-test Results (H0: exceedance rate = 1.0%):

Historical Simulation:
  N exceptions      : 86
  T obs             : 5032
  Actual rate       : 1.709
  Expected rate     : 1.0
  z-stat            : 5.0552
  p-value           : 0.0
  Reject H0         : Yes

Parametric (Normal):
  N exceptions      : 133
  T obs             : 5032
  Actual rate       : 2.643
  Expected rate     : 1.0
  z-stat            : 11.7142
  p-value           : 0.0
  Reject H0         : Yes

GARCH(1,1)-filtered:
  N exceptions      : 78
  T obs             : 5032
  Actual rate       : 1.55
  Expected rate     : 1.0
  z-stat            : 3.9217
  p-value           : 0.0001
  Reject H0         : Yes


The z-test strongly rejects the null hypothesis of correct unconditional coverage for all three VaR models. 

Historical Simulation produces 86 exceedances over 5,032 observations, corresponding to an exceedance rate of 1.71%, significantly above the theoretical 1% target (z = 5.06, p < 0.001). The Parametric Normal model performs worst, generating 133 exceedances (2.64%), more than double the expected frequency (z = 11.71, p < 0.001). This result is consistent with the evidence from Notebook 01 that portfolio returns exhibit substantial fat tails and excess kurtosis, causing the Normal assumption to systematically underestimate tail risk. 

The GARCH(1,1)-filtered approach performs best among the three methods, with 78 exceedances (1.55%). Although its exceedance rate remains statistically higher than the theoretical benchmark (z = 3.92, p < 0.001), it is materially closer to the target than either Historical Simulation or the Parametric model. 

An important observation is that all three models are rejected despite their exceedance rates differing from the target by less than two percentage points. This reflects the large sample size (5,032 trading days), which gives the test substantial statistical power. Consequently, while the z-test provides evidence of miscalibration, it offers limited information regarding the magnitude or economic significance of the deviation.

Overall, the ranking is consistent with previous findings: GARCH-filtered Historical Simulation performs best, Historical Simulation is intermediate, and the Parametric Normal model performs worst. However, the z-test only evaluates exceedance frequency and cannot determine whether exceptions occur independently through time. This motivates the use of Kupiec and Christoffersen likelihood-ratio tests in the following analysis.

### Kupiec Unconditional Coverage Test (LR$_{uc}$)

The Kupiec (1995) likelihood ratio test is the regulatory standard for backtesting VaR. It tests whether the observed exceedance rate is statistically consistent with the model's stated confidence level. Like the z-test, $LR_{uc}$ only tests **unconditional coverage** and does not detect clustering. It assumes that the exceptions are evenly distributed across the time, which means the exceptions are i.i.d.

Under $H_0$ (model well-calibrated, true exceedance prob = $p$):

$$LR_{uc} = -2\ln[(1-p)^{T-N}p^N] + 2\ln[(1-\frac{N}{T})^{T-N}(\frac{N}{T})^N] $$

$$ ==> LR_{uc} = -2\ln\left[\frac{p^N(1-p)^{T-N}}{\hat{p}^N(1-\hat{p})^{T-N}}\right] \sim \chi^2(1)$$

where $\hat{p} = N/T$ is the observed exceedance rate.

We reject $H_0$ at 5% significance if $LR_{uc} > 3.841$ (the $\chi^2(1)$ critical value at 95%).

In [5]:
def kupiec_uc(exceptions, n_obs, p=0.01, alpha=0.05):
    """
    Kupiec (1995) likelihood ratio test for unconditional coverage.
    LRuc ~ chi2(1) under H0.
    Reject H0 if LRuc > chi2(1, 1-alpha) = 3.841 at 5% level.
    """
    N = exceptions
    T = n_obs
    p_hat = N / T

    if N == 0 or N == T:
        return {
            'N exceptions': N, 'T obs': T,
            'p_hat': round(p_hat, 5),
            'LRuc': np.nan, 'p-value': np.nan,
            'Reject H0': 'N/A',
        }

    LRuc = -2 * (np.log((1-p)**(T-N) * p**N) - np.log((1-p_hat)**(T-N) * p_hat**N))

    p_value = 1 - chi2.cdf(LRuc, df=1)
    crit_val = chi2.ppf(1 - alpha, df=1)   # 3.841

    return {
        'N exceptions': N,
        'T obs': T,
        'p_hat (%)': round(p_hat * 100, 3),
        'LRuc': round(LRuc, 4),
        'p-value': round(p_value, 4),
        'Crit val': round(crit_val, 3),
        'Reject H0': 'Yes' if LRuc > crit_val else 'No',
    }

# ── Run Kupiec for all three methods ─────────────────────────────────────────
print('Kupiec Unconditional Coverage Test (H0: true exceedance prob = 1%):')
print('=' * 65)

kupiec_results = {}
for method, col in methods.items():
    var_series = risk_estimates[col].reindex(common_idx)
    exceptions = int((-port_aligned > var_series).sum())
    result = kupiec_uc(exceptions, len(common_idx))
    kupiec_results[method] = result
    print(f'\n{method}:')
    for k, v in result.items():
        print(f'  {k:<18}: {v}')

Kupiec Unconditional Coverage Test (H0: true exceedance prob = 1%):

Historical Simulation:
  N exceptions      : 86
  T obs             : 5032
  p_hat (%)         : 1.709
  LRuc              : 21.0786
  p-value           : 0.0
  Crit val          : 3.841
  Reject H0         : Yes

Parametric (Normal):
  N exceptions      : 133
  T obs             : 5032
  p_hat (%)         : 2.643
  LRuc              : 94.5577
  p-value           : 0.0
  Crit val          : 3.841
  Reject H0         : Yes

GARCH(1,1)-filtered:
  N exceptions      : 78
  T obs             : 5032
  p_hat (%)         : 1.55
  LRuc              : 13.1699
  p-value           : 0.0003
  Crit val          : 3.841
  Reject H0         : Yes


The Kupiec unconditional coverage test rejects all three VaR models at the 5% significance level.

Historical Simulation produces an exceedance rate of 1.71%, generating an $LR_{uc}$ statistic of 21.08, well above the critical value of 3.841. The Parametric Normal model performs worst, with an exceedance rate of 2.64% and an $LR_{uc}$ statistic of 94.56, indicating severe underestimation of tail risk. The GARCH(1,1)-filtered approach performs best, producing the lowest LR$_{uc}$ statistic (13.17) and an exceedance rate of 1.55%, although it still fails the unconditional coverage test.

The ranking is therefore identical to that obtained from the z-test: GARCH-filtered Historical Simulation performs best, followed by Historical Simulation, while the Parametric Normal model performs worst. This consistency is expected because both tests evaluate the same property — whether the observed exceedance frequency matches the theoretical 1% target.However, unconditional coverage alone is insufficient for model validation. A model may generate the correct number of exceedances while still producing them in clusters during periods of market stress. Such behaviour violates the independence assumption underlying VaR forecasts and motivates the Christoffersen conditional coverage test analysis.

### Christoffersen Conditional Coverage Test (LR$_{cc}$)

Christoffersen (1998) extends the Kupiec test by jointly testing two properties that a well-calibrated VaR model must satisfy:

1. **Unconditional coverage**: the exceedance rate ($\frac{N}{T}$) equals $p$
2. **Independence**: exceedances are not clustered in time

The test uses a first-order Markov chain to model the exception sequence. Define the indicator $I_t = \mathbf{1} \{\text{loss}_t > \text{VaR}_t \}$ and let $n_{ij}$ be the number of transitions from state $i$ to state $j$:

$$\hat{\pi}_{01} = \frac{n_{01}}{n_{00} + n_{01}}, \quad \hat{\pi}_{11} = \frac{n_{11}}{n_{10} + n_{11}}$$

The independence LR statistic tests $H_0: \pi_{01} = \pi_{11}$:

$$LR_{ind} = -2\ln\left[\frac{(1-\hat{\pi})^{n_{00}+n_{10}} \hat{\pi}^{n_{01}+n_{11}}} {(1-\hat{\pi}_{01})^{n_{00}}\hat{\pi}_{01}^{n_{01}} (1-\hat{\pi}_{11})^{n_{10}}\hat{\pi}_{11}^{n_{11}}}\right] \sim \chi^2(1), \quad \quad \hat{\pi} = \frac{n_{01}+n_{11}}{n_{00}+n_{01}+n_{11}+n_{10}}$$

The conditional coverage statistic combines both tests:

$$LR_{cc} = LR_{uc} + LR_{ind} \sim \chi^2(2)$$

Reject $H_0$ at 5% if $LR_{cc} > 5.991$.

In [10]:
def christoffersen_cc(exceptions_series, p=0.01, alpha=0.05):
    """
    Christoffersen (1998) conditional coverage test.
    Tests both the unconditional coverage (exceedance rate = p) and the independence of exceptions.
    LRcc ~ chisq(2) under H0.
    Reject H0 if LRcc > chi2(2, 1-alpha) = 5.991 at 5% level.
    """
    I = exceptions_series.astype(int).values
    T = len(I)
    N = I.sum()

    # Transition counts
    n00 = ((I[:-1] == 0) & (I[1:] == 0)).sum()
    n01 = ((I[:-1] == 0) & (I[1:] == 1)).sum()
    n10 = ((I[:-1] == 1) & (I[1:] == 0)).sum()
    n11 = ((I[:-1] == 1) & (I[1:] == 1)).sum()

    # Transition probabilities
    pi_hat = N/T
    pi_hat_ind = (n01 + n11) / (n00 + n01 + n10 + n11)
    pi_01 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0
    pi_11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0

    # LRuc
    LRuc  = -2 * (np.log((1 - p) ** (T - N) * p ** N) - np.log((1 - pi_hat) ** (T - N) * pi_hat ** N)) if 0 < pi_hat < 1 else np.nan

    # LRind
    try:
        L_restricted = (1 - pi_hat_ind) ** (n00 + n10) * pi_hat_ind ** (n01 + n11)
        L_unrestricted = ((1 - pi_01) ** n00 * pi_01 ** n01 * (1 - pi_11) ** n10 * pi_11 ** n11)
        LRind = -2 * (np.log(L_restricted) - np.log(L_unrestricted))
    except Exception:
        LRind = np.nan

    LRcc = LRuc + LRind
    pval_uc = 1 - chi2.cdf(LRuc,  df=1)
    pval_ind = 1 - chi2.cdf(LRind, df=1)
    pval_cc = 1 - chi2.cdf(LRcc,  df=2)
    crit_uc = chi2.ppf(1 - alpha, df=1)   # 3.841
    crit_cc = chi2.ppf(1 - alpha, df=2)   # 5.991

    return {
        'N exceptions': int(N),
        'T obs' : T,
        'n00, n01' : f'{n00}, {n01}',
        'n10, n11' : f'{n10}, {n11}',
        'pi_hat01' : round(pi_01, 4),
        'pi_hat11' : round(pi_11, 4),
        'LRuc' : round(LRuc,  4),
        'LRind' : round(LRind, 4),
        'LRcc' : round(LRcc,  4),
        'p-val LRuc' : round(pval_uc,  4),
        'p-val LRind' : round(pval_ind, 4),
        'p-val LRcc' : round(pval_cc,  4),
        'Reject LRuc' : 'Yes' if LRuc  > crit_uc else 'No',
        'Reject LRind': 'Yes' if LRind > crit_uc else 'No',
        'Reject LRcc' : 'Yes' if LRcc  > crit_cc else 'No',
    }

# ── Run Christoffersen for all three methods ──────────────────────────────────
print('Christoffersen Conditional Coverage Test:')
print('=' * 65)

cc_results = {}
for method, col in methods.items():
    var_series = risk_estimates[col].reindex(common_idx)
    exceed_series = -port_aligned > var_series
    result = christoffersen_cc(exceed_series)
    cc_results[method] = result
    print(f'\n{method}:')
    for k, v in result.items():
        print(f'  {k:<18}: {v}')

Christoffersen Conditional Coverage Test:

Historical Simulation:
  N exceptions      : 86
  T obs             : 5032
  n00, n01          : 4865, 80
  n10, n11          : 80, 6
  pi_hat01          : 0.0162
  pi_hat11          : 0.0698
  LRuc              : 21.0786
  LRind             : 8.3161
  LRcc              : 29.3947
  p-val LRuc        : 0.0
  p-val LRind       : 0.0039
  p-val LRcc        : 0.0
  Reject LRuc       : Yes
  Reject LRind      : Yes
  Reject LRcc       : Yes

Parametric (Normal):
  N exceptions      : 133
  T obs             : 5032
  n00, n01          : 4778, 120
  n10, n11          : 120, 13
  pi_hat01          : 0.0245
  pi_hat11          : 0.0977
  LRuc              : 94.5577
  LRind             : 16.4736
  LRcc              : 111.0313
  p-val LRuc        : 0.0
  p-val LRind       : 0.0
  p-val LRcc        : 0.0
  Reject LRuc       : Yes
  Reject LRind      : Yes
  Reject LRcc       : Yes

GARCH(1,1)-filtered:
  N exceptions      : 78
  T obs             : 5032
 

The Christoffersen conditional coverage test reveals substantial differences in the independence properties of the three VaR models.

**Historical Simulation** exhibits significant exception clustering (LRind = 8.32, p = 0.004), with the probability of an exceedance following another exceedance ($ \hat{\pi}_{1,1} = 6.98 \%$) substantially exceeding the probability of an exceedance following a non-exceedance day ($ \hat{\pi}_{0,1} = 1.62 \%$). This pattern is consistent with the slow adaptation of historical simulation to changing volatility regimes.

**Parametric Normal** shows the strongest evidence of dependence (LRind = 16.47), reflecting its inability to capture time-varying volatility and volatility clustering.

In contrast, the **GARCH(1,1)-filtered VaR model** is the only specification for which the independence hypothesis cannot be rejected (LRind = 1.95, p = 0.162). Although exceedances remain somewhat more likely after previous exceedances ($ \hat{\pi}_{1,1} = 3.85 \%$ versus $ \hat{\pi}_{0,1} = 1.51 \%$), the difference is not statistically significant at conventional confidence levels. This suggests that the conditional volatility dynamics embedded in the GARCH framework substantially reduce exception clustering.

While the conditional coverage test rejects all three models, the decomposition is informative. For Historical Simulation and Parametric Normal, rejection is driven by both incorrect exceedance frequency and dependence in exception timing. For GARCH-filtered VaR, rejection is driven primarily by unconditional coverage, indicating that the model captures volatility persistence reasonably well but still underestimates overall tail risk.

## PIT-Based Backtesting

Exceedance-based tests only use information at the VaR threshold — they discard the shape of the loss distribution elsewhere. **PIT-based backtesting** uses the full forecast distribution.

The **Probability Integral Transform** for day $t+1$ is:

$$\text{PIT}_{t+1} = F_t(P\&L_{t+1}) = \mathbb{P}(X \leq P\&L_{t+1})$$

where $F_t(\cdot)$ is the daily forecast CDF from the previous day's model.

In PIT backtesting, if the risk is correctly modeled:
1. The uncondistional coverage means that the time series of $\text{PIT}_{t}$ should be uniformly distributed;
2. The independence means that series of $\text{PIT}_{t}$ are independent identical distribution.

$$\text{PIT}_t \overset{i.i.d.}{\sim} \text{Uniform}(0, 1)$$

We test this using three goodness-of-fit tests:
- **K-S test**: maximum deviation from uniformity
- **Anderson-Darling**: weighted K-S, emphasises tails
- **Cramér-von Mises**: mean squared deviation from uniformity

### Compute PIT series

First, we compute the PIT series for each VaR model. Unlike exceedance-based tests, PIT-based backtesting requires the full one-step-ahead forecast distribution, not only the VaR quantile.

For Historical Simulation, the forecast distribution is approximated by the rolling empirical CDF based on the previous 250 observations. For the Parametric Normal model, the PIT is computed using the rolling normal CDF with the corresponding forecast mean and volatility. For the GARCH(1,1)- filtered model, the PIT is computed from the standardized return/loss using the one-step-ahead conditional volatility forecast, under the normal innovation assumption.

In [16]:
def compute_pit_hs(port_ret, window=250):
    pit = pd.Series(index=port_ret.index, dtype=float)
    for t in range(window, len(port_ret)):
        window_ret = port_ret.iloc[t - window:t].values
        realised = port_ret.iloc[t]
        pit.iloc[t] = (window_ret <= realised).mean()
    return pit.dropna()


def compute_pit_parametric(port_ret, window=250):
    pit = pd.Series(index=port_ret.index, dtype=float)
    for t in range(window, len(port_ret)):
        window_ret = port_ret.iloc[t - window:t]
        mu = window_ret.mean()
        sigma = window_ret.std(ddof=1)
        realised = port_ret.iloc[t]
        pit.iloc[t] = stats.norm.cdf(realised, loc=mu, scale=sigma)
    return pit.dropna()


def compute_pit_garch(port_ret, window=250):
    """
    PIT for GARCH(1,1):
    Fit GARCH on window, get one-step-ahead sigma forecast,
    """
    pit = pd.Series(index=port_ret.index, dtype=float)
    for t in range(window, len(port_ret)):
        window_ret = port_ret.iloc[t - window:t] * 100
        realised = port_ret.iloc[t]
        try:
            model = arch_model(
                window_ret, vol='Garch', p=1, q=1,
                mean='Constant', dist='normal'
            )
            result = model.fit(disp='off', show_warning=False)
            mu_hat = result.params['mu'] / 100
            forecast = result.forecast(horizon=1)
            sigma_next = np.sqrt(forecast.variance.values[-1, 0]) / 100
            pit.iloc[t] = stats.norm.cdf(realised, loc=mu_hat, scale=sigma_next)
        except Exception:
            # Fallback to parametric if GARCH fails
            window_ret_orig = port_ret.iloc[t - window:t]
            mu = window_ret_orig.mean()
            sigma = window_ret_orig.std(ddof=1)
            pit.iloc[t] = stats.norm.cdf(realised, loc=mu, scale=sigma)
    return pit.dropna()


print('Computing PIT series')
print('HS PIT')
pit_hs = compute_pit_hs(port_aligned)
print('Parametric Normal PIT')
pit_param = compute_pit_parametric(port_aligned)
print('GARCH PIT')
pit_garch = compute_pit_garch(port_aligned)


print('\nUnder H0: mean = 0.500, std = 0.289 (Uniform[0,1])')
print('-' * 50)
for name, pit in [('HS', pit_hs), ('Parametric', pit_param), ('GARCH', pit_garch)]:
    print(f'\n{name} PIT ({len(pit)} obs):')
    print(f'mean = {pit.mean():.4f}  '
          f'std = {pit.std():.4f}  '
          f'min = {pit.min():.4f}  '
          f'max = {pit.max():.4f}')

Computing PIT series
HS PIT
Parametric Normal PIT
GARCH PIT

Under H0: mean = 0.500, std = 0.289 (Uniform[0,1])
--------------------------------------------------

HS PIT (4782 obs):
mean = 0.4992  std = 0.2891  min = 0.0000  max = 1.0000

Parametric PIT (4782 obs):
mean = 0.5020  std = 0.2657  min = 0.0000  max = 1.0000

GARCH PIT (4782 obs):
mean = 0.4946  std = 0.2783  min = 0.0000  max = 1.0000


All three PIT series are centred close to the theoretical Uniform(0,1) mean of 0.5. HS has a standard deviation almost identical to the uniform benchmark (0.289), while Parametric Normal and GARCH show lower dispersion, suggesting potential deviations from uniformity. Formal goodness-of-fit tests are reported below.

### PIT Histogram

We plot the empirical distribution of PIT values as a histogram and compare it to the theoretical Uniform(0,1) density. Deviations from a flat histogram indicate model misspecification:

- **Excess mass near 0**: model underestimates left tail risk (too many large losses assigned low probability)
- **Excess mass near 1**: model overestimates left tail risk
- **U-shape**: model underestimates volatility (fat tails)
- **Hump in centre**: model overestimates volatility

In [20]:
n_bins = 20
bin_width = 1 / n_bins
uniform_height = len(pit_hs) * bin_width

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'PIT — Historical Simulation',
        'PIT — Parametric (Normal)',
        'PIT — GARCH(1,1)',
    ],
    horizontal_spacing=0.08,
)

for col_idx, (pit, label) in enumerate(
    [(pit_hs, 'HS'),
     (pit_param, 'Parametric'),
     (pit_garch, 'GARCH')],
    start=1
):
    fig.add_trace(
        go.Histogram(
            x = pit,
            nbinsx = n_bins,
            name = label,
            marker = dict(color='steelblue', opacity=0.7),
            showlegend= False,
        ),
        row=1, col=col_idx,
    )
    fig.add_hline(
        y = uniform_height,
        line_dash = 'dash',
        line_color = 'red',
        line_width = 1.5,
        annotation_text = 'Uniform(0,1)',
        annotation_position= 'right',
        row=1, col=col_idx,
    )

fig.update_xaxes(title_text='PIT value', range=[0, 1])
fig.update_yaxes(title_text='Frequency')

The PIT histograms provide a visual assessment of forecast distribution calibration.

The **Historical Simulation PIT histogram** is broadly consistent with a Uniform(0,1) distribution, indicating reasonable calibration of the forecast distribution. Small deviations from uniformity remain, particularly in the tails, reflecting the inherent limitations of a finite rolling historical window.

The **Parametric Normal model** exhibits a pronounced hump around 0.5 and fewer observations in the tails. This pattern indicates that realised returns occur too frequently near the centre of the forecast distribution, consistent with an overly restrictive Normal assumption and inadequate tail modelling.

The **GARCH(1,1) PIT** distribution remains more dispersed than the Parametric Normal model and appears closer to uniformity. This suggests that incorporating time-varying volatility improves distributional calibration, although some deviations from uniformity remain.

These visual findings are broadly consistent with the exceedance-based backtesting results, where the GARCH model exhibited the strongest independence properties and the Parametric Normal model showed the largest degree of misspecification.

### 3.3 Goodness-of-Fit Tests

We formally test whether each PIT series is Uniform(0,1) using three complementary statistics:

**Kolmogorov-Smirnov (K-S)**:

K-S test is a non-parametric test, comparing emperical distribution function of a sample with reference dstribution function (Unif(0,1)). Quantifies the distance between the emperical CDF of the sample and the CDF of the reference distribution. The K-S test statistic is:

$$D = \sup_x |F_n(x) - F_U(x)|$$

The limitation of K-S test is that it lacks of sensitivity at the extremes of the distribution. While the test is effective in detecting differences in central masses of the two distributions, it is not reliable when it comes to comparison of the tails.

**Anderson-Darling (A-D)**:

A-D test is a modified K-S test, tries to correct for the limitation of K-S we mentioned above. A-D test puts more weight on the tails of the distribution, yielding more power in the presence of distribution biases. Therefore, any misspecification in the tail of the distribution of risk model will be better identified by A-D. The A-D test statistic is:

$$A^2 = -n - \frac{1}{n}\sum_{j=1}^{n}(2j-1)[\ln z_j + \ln(1-z_{n+1-j})]$$

**Cramér-von Mises (C-M)**:

C-M test is a variation of K-S test. Rather than using supremum, C-M relies on the mean-squared deviation of the distribution to balance sensitivity across the distribution. The C-M test statistic is:

$$W^2 = \sum_{j=1}^{n}\left(z_j - \frac{2j-1}{2n}\right)^2 + \frac{1}{12n}$$

All tests use $H_0$: PIT $\sim$ Uniform(0,1). Rejection will indicate distributional misspecification in the forecast model.

In [22]:
def pit_gof_tests(pit_series, name, alpha=0.05):
    z = np.sort(pit_series.values)
    n = len(z)

    # K-S test
    ks_stat, ks_pval = kstest(z, 'uniform')

    # A-D test
    j = np.arange(1, n + 1)
    # Clip to avoid log(0)
    z_clip = np.clip(z, 1e-10, 1 - 1e-10)
    ad_stat = -n - (1/n) * np.sum((2*j - 1) * (np.log(z_clip) + np.log(1 - z_clip[::-1])))
    # Critical values for A-D uniform test at common significance levels
    ad_crit = {0.10: 1.933, 0.05: 2.492, 0.025: 3.070, 0.01: 3.857}
    ad_reject = 'Yes' if ad_stat > ad_crit[alpha] else 'No'

    # C-M test
    cm_stat = np.sum((z - (2*j - 1) / (2*n))**2) + 1 / (12*n)
    # Critical values for C-M uniform test at common significance levels
    cm_crit = {0.10: 0.347, 0.05: 0.461, 0.025: 0.581, 0.01: 0.743}
    cm_reject = 'Yes' if cm_stat > cm_crit[alpha] else 'No'

    print(f'\n{name}  (n={n}):')
    print(f' K-S  stat = {ks_stat:.4f}   p-value = {ks_pval:.4f}   Reject H0: {"Yes" if ks_pval < alpha else "No"}')
    print(f' A-D  stat = {ad_stat:.4f}   crit({int(alpha*100)}%) = {ad_crit[alpha]}   Reject H0: {ad_reject}')
    print(f' C-M  stat = {cm_stat:.4f}   crit({int(alpha*100)}%) = {cm_crit[alpha]}   Reject H0: {cm_reject}')

    return {
        'Method' : name,
        'n' : n,
        'KS stat' : round(ks_stat, 4),
        'KS p-value' : round(ks_pval, 4),
        'KS reject' : 'Yes' if ks_pval < alpha else 'No',
        'AD stat' : round(ad_stat, 4),
        'AD crit (5%)' : ad_crit[alpha],
        'AD reject' : ad_reject,
        'CM stat' : round(cm_stat, 4),
        'CM crit (5%)' : cm_crit[alpha],
        'CM reject' : cm_reject,
    }

print('PIT Goodness-of-Fit Tests (H0: PIT ~ Uniform(0,1)):')
print('=' * 65)

gof_results = []
for pit, name in [
    (pit_hs, 'Historical Simulation'),
    (pit_param, 'Parametric (Normal)'),
    (pit_garch, 'GARCH(1,1)-filtered'),
]:
    result = pit_gof_tests(pit, name)
    gof_results.append(result)

gof_df = pd.DataFrame(gof_results).set_index('Method')
print(f'\nSummary table:')
print(gof_df[['KS stat', 'KS reject',
              'AD stat', 'AD reject',
              'CM stat', 'CM reject']].to_string())

PIT Goodness-of-Fit Tests (H0: PIT ~ Uniform(0,1)):

Historical Simulation  (n=4782):
 K-S  stat = 0.0109   p-value = 0.6204   Reject H0: No
 A-D  stat = 5.4345   crit(5%) = 2.492   Reject H0: Yes
 C-M  stat = 0.0654   crit(5%) = 0.461   Reject H0: No

Parametric (Normal)  (n=4782):
 K-S  stat = 0.0647   p-value = 0.0000   Reject H0: Yes
 A-D  stat = 40.0280   crit(5%) = 2.492   Reject H0: Yes
 C-M  stat = 6.5175   crit(5%) = 0.461   Reject H0: Yes

GARCH(1,1)-filtered  (n=4782):
 K-S  stat = 0.0340   p-value = 0.0000   Reject H0: Yes
 A-D  stat = 13.0811   crit(5%) = 2.492   Reject H0: Yes
 C-M  stat = 1.7835   crit(5%) = 0.461   Reject H0: Yes

Summary table:
                       KS stat KS reject  AD stat AD reject  CM stat CM reject
Method                                                                        
Historical Simulation   0.0109        No   5.4345       Yes   0.0654        No
Parametric (Normal)     0.0647       Yes  40.0280       Yes   6.5175       Yes
GARCH(1,1)-fil

The PIT goodness-of-fit tests reveal clear differences in distributional calibration across the three VaR models. 

**Historical Simulation** passes the K-S and C-M tests but is rejected by the A-D test, indicating that the overall PIT distribution is close to Uniform(0,1) while residual misspecification remains in the tails. The **Parametric Normal model** is strongly rejected by all three tests, confirming substantial deviations from Uniform(0,1). This finding is consistent with the PIT histogram and previous exceedance-based backtests, which showed that the Normal assumption fails to capture the heavy-tailed nature of portfolio returns. The **GARCH(1,1)-filtered model** improves upon the Parametric Normal specification but remains rejected by all three PIT tests. Combined with the earlier Christoffersen results, this suggests that GARCH captures volatility clustering and exception independence reasonably well, but the Normal innovation assumption remains insufficient to fully describe the return distribution. 

The divergence between HS passing K-S and C-M but failing A-D is particularly informative. Since A-D applies greater weight to the tails of the distribution, this pattern suggests that HS's misspecification is concentrated in the extreme left tail — precisely the region relevant for risk management. This finding is consistent with the exceedance
clustering result from the Christoffersen test: HS underestimates tail risk during stress regimes when the 250-day window contains insufficient crisis observations.

Overall, Historical Simulation provides the best PIT calibration among the three approaches, while Parametric Normal performs the worst.

## Regime-Conditional Backtesting

We repeat the exceedance-based backtests separately within each HMM regime. This reveals whether model failures are uniformly distributed across market environments or concentrated in specific regimes,  help to better understand when each model breaks down.

A well-calibrated model should achieve approximately 1% exceedance rate in **every** regime, not just on average.

In [25]:
def backtest_summary(exceed_series, n_obs, p=0.01):
    """
    Compute z-test and Kupiec LRuc for a given exception series.
    """
    N = int(exceed_series.sum())
    p_hat = N / n_obs
    # z-test
    z_stat = (p_hat - p) / np.sqrt(p * (1 - p) / n_obs)
    z_pval = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    # Kupiec
    if 0 < p_hat < 1:
        LRuc  = -2 * (np.log((1 - p) ** (n_obs - N) * p ** N) - np.log((1 - p_hat) ** (n_obs - N) * p_hat ** N))
        lr_pval = 1 - chi2.cdf(LRuc, df=1)
    else:
        LRuc = np.nan
        lr_pval = np.nan

    return {
        'N except' : N,
        'T obs' : n_obs,
        'Rate (%)' : round(p_hat * 100, 2),
        'z-stat' : round(z_stat, 3),
        'z p-val' : round(z_pval, 4),
        'LRuc' : round(LRuc, 3) if not np.isnan(LRuc) else 'N/A',
        'LRuc p-val' : round(lr_pval, 4) if not np.isnan(lr_pval) else 'N/A',
        'Reject (5%)' : 'Yes' if z_pval < 0.05 else 'No',
    }

regimes = ['expansion', 'normal', 'stress']

print('Regime-Conditional Backtesting (99% VaR):')
print('=' * 70)
print(f'Expected exceedance rate: 1.00% in every regime\n')

regime_bt_results = {}

for method, col in methods.items():
    print(f'\n── {method} ──')
    var_series = risk_estimates[col].reindex(common_idx)
    exceed_all = -port_aligned > var_series
    method_results = {}

    for regime in regimes:
        mask = regime_aligned == regime
        exceed_reg = exceed_all[mask]
        n_reg = int(mask.sum())
        result = backtest_summary(exceed_reg, n_reg)
        method_results[regime] = result
        reject_flag  = '[REJECT]' if result['Reject (5%)'] == 'Yes' else '[OK]'
        print(f'{regime.capitalize():<12} '
              f'N={result["N except"]:3d}/{result["T obs"]:4d}  '
              f'rate={result["Rate (%)"]:.2f}%  '
              f'z={result["z-stat"]:6.3f}  '
              f'LRuc={result["LRuc"]}  '
              f'{reject_flag}')

    regime_bt_results[method] = method_results

# Stress-regime focus
print('\n\nStress Regime Detail (most critical for risk management):')
print('-' * 70)
for method in methods:
    r = regime_bt_results[method]['stress']
    print(f'{method:<30}: rate={r["Rate (%)"]:.2f}%  LRuc={r["LRuc"]}  Reject={r["Reject (5%)"]}')

Regime-Conditional Backtesting (99% VaR):
Expected exceedance rate: 1.00% in every regime


── Historical Simulation ──
Expansion    N= 12/2434  rate=0.49%  z=-2.514  LRuc=7.77  [REJECT]
Normal       N= 35/1839  rate=1.90%  z= 3.893  LRuc=11.98  [REJECT]
Stress       N= 39/ 759  rate=5.14%  z=11.459  LRuc=66.177  [REJECT]

── Parametric (Normal) ──
Expansion    N= 16/2434  rate=0.66%  z=-1.699  LRuc=3.284  [OK]
Normal       N= 60/1839  rate=3.26%  z= 9.752  LRuc=59.643  [REJECT]
Stress       N= 57/ 759  rate=7.51%  z=18.025  LRuc=134.352  [REJECT]

── GARCH(1,1)-filtered ──
Expansion    N= 20/2434  rate=0.82%  z=-0.884  LRuc=0.832  [OK]
Normal       N= 39/1839  rate=2.12%  z= 4.830  LRuc=17.651  [REJECT]
Stress       N= 19/ 759  rate=2.50%  z= 4.162  LRuc=12.223  [REJECT]


Stress Regime Detail (most critical for risk management):
----------------------------------------------------------------------
Historical Simulation         : rate=5.14%  LRuc=66.177  Reject=Yes
Parametric (Normal

The results reveal that VaR model performance is highly regime dependent. 

All three models perform reasonably well during **Expansion regimes**, with exceedance rates close to the theoretical 1% target. The GARCH-filtered model performs best, producing an exceedance rate of 0.82%, while both the Parametric Normal and GARCH models pass the Kupiec test. Model performance deteriorates substantially in **Normal and Stress regimes**. Historical Simulation exhibits exceedance rates of 1.90% and 5.14%, respectively, indicating that the rolling historical window fails to adapt quickly enough when market conditions deteriorate. The Parametric Normal model performs worst overall. Exceedance rates rise to 3.26% in the Normal regime and 7.51% in the Stress regime, confirming that the Normal assumption severely underestimates tail risk during periods of elevated volatility. The GARCH(1,1)-filtered model provides the strongest performance across regimes. Although it still fails unconditional coverage tests in the Normal and Stress states, exceedance rates remain substantially lower than those of the Historical Simulation and Parametric Normal models. In the **Stress regime**, the exceedance rate falls from 7.51% under the Parametric model and 5.14% under Historical Simulation to 2.50% under GARCH. 

These findings indicate that the poor full-sample backtesting results are not uniformly distributed through time. Instead, model failures are concentrated in adverse market regimes, particularly during periods of elevated volatility and market stress. This highlights the importance of regime-aware risk measurement and demonstrates that volatility modelling provides the greatest benefit precisely when risk management is most critical.

## Backtesting Summary

We consolidate all backtesting results into a single comparison table to facilitate model selection and identify each model's key failure mode.

In [33]:
summary_rows = []

for method, col in methods.items():
    var_series = risk_estimates[col].reindex(common_idx)
    exceed_series = -port_aligned > var_series
    N = int(exceed_series.sum())
    T = len(common_idx)

    # Full-sample stats
    p_hat = N / T
    z_stat = (p_hat - 0.01) / np.sqrt(0.01 * 0.99 / T)
    LRuc = -2 * (np.log(0.99 ** (T - N) * 0.01 ** N) - np.log((1 - p_hat) ** (T - N) * p_hat ** N))

    # Christoffersen independence
    I = exceed_series.astype(int).values
    n11 = ((I[:-1] == 1) & (I[1:] == 1)).sum()
    n10 = ((I[:-1] == 1) & (I[1:] == 0)).sum()

    # PIT A-D stat (tail-sensitive)
    pit_map = {
        'Historical Simulation': pit_hs,
        'Parametric (Normal)' : pit_param,
        'GARCH(1,1)-filtered' : pit_garch,
    }
    pit = pit_map[method]
    z_pit = np.sort(pit.values)
    n_pit = len(z_pit)
    j = np.arange(1, n_pit + 1)
    z_clip = np.clip(z_pit, 1e-10, 1 - 1e-10)
    ad_stat = -n_pit - (1/n_pit) * np.sum((2*j - 1) * (np.log(z_clip) + np.log(1 - z_clip[::-1])))

    # Stress regime exceedance rate
    stress_mask = regime_aligned == 'stress'
    stress_exceed = exceed_series[stress_mask].sum()
    stress_n = stress_mask.sum()
    stress_rate = stress_exceed / stress_n * 100

    summary_rows.append({
        'Model' : method,
        'Exceedances' : N,
        'Full rate (%)' : round(p_hat * 100, 2),
        'z-stat' : round(z_stat, 2),
        'LRuc' : round(LRuc, 2),
        'Reject LRuc' : 'Yes' if LRuc > 3.841 else 'No',
        'LRind' : round(cc_results[method]['LRind'], 2),
        'Reject LRind' : cc_results[method]['Reject LRind'],
        'LRcc' : round(cc_results[method]['LRcc'], 2),
        'Reject LRcc' : cc_results[method]['Reject LRcc'],
        'AD stat (PIT)' : round(ad_stat, 2),
        'AD reject (PIT)' : 'Yes' if ad_stat > 2.492 else 'No',
        'Stress rate (%)' : round(stress_rate, 2),
        'Primary failure' : (
            'Frequency + Clustering + Tails' if method == 'Parametric (Normal)'
            else 'Frequency + Clustering' if method == 'Historical Simulation'
            else 'Frequency only'
        ),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print('Backtesting Summary — All Methods:')
print('=' * 210)
print(summary_df.to_string())

print('\nRecall:')
print(' LRuc crit (5%) = 3.841  (chisq(1))')
print(' LRcc crit (5%) = 5.991  (chisq(2))')
print(' AD crit (5%) = 2.492')

Backtesting Summary — All Methods:
                       Exceedances  Full rate (%)  z-stat   LRuc Reject LRuc  LRind Reject LRind    LRcc Reject LRcc  AD stat (PIT) AD reject (PIT)  Stress rate (%)                 Primary failure
Model                                                                                                                                                                                               
Historical Simulation           86           1.71    5.06  21.08         Yes   8.32          Yes   29.39         Yes           5.43             Yes             5.14          Frequency + Clustering
Parametric (Normal)            133           2.64   11.71  94.56         Yes  16.47          Yes  111.03         Yes          40.03             Yes             7.51  Frequency + Clustering + Tails
GARCH(1,1)-filtered             78           1.55    3.92  13.17         Yes   1.95           No   15.12         Yes          13.08             Yes             2.50             

Across all backtesting dimensions, the GARCH(1,1)-filtered VaR model provides the strongest overall performance. It is the only model that passes the Christoffersen independence test and produces the lowest stress-regime exceedance rate (2.50%). Historical Simulation achieves reasonable overall PIT calibration but fails to adequately capture extreme market conditions, leading to exception clustering and elevated exceedance rates during stress periods. The Parametric Normal model performs worst across all diagnostics, showing substantial underestimation of tail risk, strong exception clustering, and the highest stress-regime exceedance rate. 

These results suggest that modelling time-varying volatility is essential for risk measurement. However, the remaining PIT and coverage failures indicate that volatility dynamics alone are insufficient. Future work could extend the framework using heavy-tailed innovations (e.g. Student-t GARCH) to better capture the distribution of extreme losses.

In [35]:
# Save outputs 

# 3 methods exceedance series
exceed_df = pd.DataFrame({
    'hs_exceed'   : (-port_aligned > risk_estimates['hs_var99'].reindex(common_idx)).astype(int),
    'param_exceed': (-port_aligned > risk_estimates['param_var99'].reindex(common_idx)).astype(int),
    'garch_exceed': (-port_aligned > risk_estimates['garch_var99'].reindex(common_idx)).astype(int),
})
exceed_df.to_csv(OUT_BT / 'exceedance_series.csv')

# PIT series
pit_df = pd.DataFrame({
    'pit_hs'   : pit_hs,
    'pit_param': pit_param,
    'pit_garch': pit_garch,
})
pit_df.to_csv(OUT_BT / 'pit_series.csv')

# Backtesting summary table
summary_df.to_csv(OUT_BT / 'backtest_summary.csv')

# Regime-conditional backtesting results
regime_bt_rows = []
for method, regime_dict in regime_bt_results.items():
    for regime, result in regime_dict.items():
        row = {'method': method, 'regime': regime}
        row.update(result)
        regime_bt_rows.append(row)

regime_bt_df = pd.DataFrame(regime_bt_rows)
regime_bt_df.to_csv(OUT_BT / 'regime_backtest.csv', index=False)

## Key Findings

1. **All three models fail full-sample unconditional coverage**, with exceedance rates exceeding the 1% target. Parametric Normal performs worst (2.64%), GARCH best (1.55%). The z-test and Kupiec LRuc reject all three models at 5% significance.

2. **Exception clustering is a critical differentiator.** Historical Simulation and Parametric Normal both fail the Christoffersen independence test. Exceptions occur in runs during stress periods, indicating the models fail to adapt to volatility regime changes. GARCH(1,1) is the only model that passes LRind (p = 0.162), confirming its time-varying volatility forecast breaks up exception clusters.

3. **PIT analysis reveals the nature of distributional misspecification.** HS passes K-S and C-M but fails A-D, which indicates misspecification is concentrated in the tails. Parametric Normal fails all three tests by wide margins, indicating global distributional failure. GARCH improves on Parametric but still fails, pointing to inadequate tail modelling under Normal innovations.

4. **Model failures are concentrated in stress regimes.** Stress-period exceedance rates are 5.14% (HS), 7.51% (Parametric), and 2.50% (GARCH), all far above the 1% target. Expansion-period rates are near or below 1% for all models, confirming that backtesting failures are regime-dependent rather than uniformly distributed through time.

5. **GARCH(1,1)-filtered HS is the best-performing model** across all backtesting dimensions: lowest exceedance rate, only model passing LRind, lowest stress-regime failure rate. Its remaining failures stem from unconditional coverage and residual distributional misspecification, suggesting that volatility dynamics are captured reasonably well, while the Normal innovation assumption remains too restrictive for extreme losses. Recalibration to a higher confidence level or Student-t innovations would address the residual misspecification.

*Next -> Notebook 04: Dependence & Copula Analysis, moving beyond marginal VaR estimation to model cross-asset dependence, tail co-movement, and portfolio loss aggregation under different market regimes.*
